# 06 — Structured Streaming + Classic Interview Coding Problems

A structured-streaming primer (micro-batches, output modes, watermarking) followed by the hands-on coding problems that show up again and again in data engineer interviews: word count, top-N, dedup, sessionization, and nth-highest-value-per-group.

> **Setup note:** these notebooks are written but **not executed** — PySpark is not
> installed in this environment. To run them locally:
>
> ```bash
> python -m venv .venv && source .venv/bin/activate
> pip install pyspark==3.5.1
> # Java 11/17 must be on PATH (java -version)
> jupyter notebook
> ```
>
> Everything below is correct, runnable PySpark — read it as a reference and run
> cell-by-cell once your environment is set up.

## 1. Structured Streaming — the mental model

Structured Streaming treats a stream as an **unbounded table** that keeps growing — you write the *same* DataFrame/SQL logic you'd write for a batch job, and Spark incrementally re-runs it as new data arrives. Under the hood (in the default micro-batch mode) it processes new data in small batches on a trigger interval rather than truly row-at-a-time (a separate low-latency "continuous processing" mode exists but supports a much smaller operator set and is rarely used in practice).

**Output modes** — how results are written to the sink on each trigger:
- **Append** — only new rows since the last trigger. Only valid when rows, once emitted, will never be updated (e.g. no aggregation, or an aggregation with a watermark that guarantees a group is "final").
- **Update** — only rows that changed since the last trigger.
- **Complete** — the entire updated result table, every trigger. Only feasible for aggregations with a bounded number of groups.

In [ ]:
# Streaming code is written but not run here (no live source in this environment).
# Pattern for reading from Kafka and writing aggregated results to a sink:

# raw_stream = (
#     spark.readStream
#     .format("kafka")
#     .option("kafka.bootstrap.servers", "localhost:9092")
#     .option("subscribe", "clickstream")
#     .load()
# )
#
# query = (
#     raw_stream
#     .selectExpr("CAST(value AS STRING) as json_str")
#     .writeStream
#     .format("console")
#     .outputMode("append")
#     .trigger(processingTime="10 seconds")
#     .start()
# )
# query.awaitTermination()

## 2. Watermarking and windowed aggregations

In streaming, "how long do we wait for late-arriving data before we stop updating a time-window's aggregate and can safely drop its state?" is answered by a **watermark**: `withWatermark("event_time", "10 minutes")` tells Spark "once you've seen an event_time of `T`, you can discard state for windows ending before `T - 10 minutes`" — bounding the state Spark must keep in memory forever, at the cost of dropping data that arrives later than the threshold.

In [ ]:
# window() buckets rows into fixed time intervals on event time (not wall-clock
# processing time), which is what makes results reproducible/correct regardless
# of processing delays:
#
# from pyspark.sql.functions import window
#
# windowed_counts = (
#     events
#     .withWatermark("event_time", "10 minutes")
#     .groupBy(window("event_time", "5 minutes"), "event_type")
#     .count()
# )
# windowed_counts.writeStream.outputMode("update").format("console").start()

## 3. Classic coding problems

These are worked with plain (batch) DataFrames — the same patterns apply verbatim inside a streaming query.

### Problem 1 — Word count

The "hello world" of distributed data processing: split lines into words, count occurrences.

In [ ]:
lines = spark.createDataFrame(
    [("the quick brown fox",), ("the lazy dog sleeps",), ("the fox jumps over the dog",)],
    ["line"],
)

from pyspark.sql.functions import split, explode, lower

word_counts = (
    lines
    .select(explode(split(lower(col("line")), " ")).alias("word"))
    .groupBy("word")
    .count()
    .orderBy(col("count").desc())
)
word_counts.show()

### Problem 2 — Deduplicate, keeping the latest record per key

"You have multiple updates per `user_id` with a `updated_at` timestamp — keep only the most recent row per user." Solved with the same `row_number()` + `filter` pattern as top-N-per-group (notebook 04) — **not** `dropDuplicates()`, which keeps an arbitrary row, not the latest.

In [ ]:
updates = spark.createDataFrame(
    [
        (1, "alice@x.com", "2024-01-01"),
        (1, "alice@new.com", "2024-03-05"),
        (2, "bob@x.com", "2024-02-10"),
        (2, "bob@older.com", "2024-01-20"),
    ],
    ["user_id", "email", "updated_at"],
)

w = Window.partitionBy("user_id").orderBy(col("updated_at").desc())
latest_per_user = (
    updates.withColumn("rn", row_number().over(w))
    .filter(col("rn") == 1)
    .drop("rn")
)
latest_per_user.show()

### Problem 3 — Sessionization

"Group a user's events into sessions, where a new session starts after 30+ minutes of inactivity." Classic pattern: compute the gap from the *previous* event with `lag()`, flag a new-session boundary wherever the gap exceeds the threshold, then a **running sum of the boundary flags** gives each row its session id.

In [ ]:
from pyspark.sql.functions import unix_timestamp, to_timestamp, sum as spark_sum3

events = spark.createDataFrame(
    [
        ("u1", "2024-01-01 10:00:00"),
        ("u1", "2024-01-01 10:05:00"),
        ("u1", "2024-01-01 10:50:00"),   # >30 min gap -> new session
        ("u1", "2024-01-01 10:55:00"),
        ("u2", "2024-01-01 09:00:00"),
    ],
    ["user_id", "event_time"],
).withColumn("event_time", to_timestamp("event_time"))

w = Window.partitionBy("user_id").orderBy("event_time")
gap_seconds = unix_timestamp(col("event_time")) - unix_timestamp(lag("event_time", 1).over(w))

sessions = (
    events
    .withColumn("is_new_session", when((gap_seconds > 30 * 60) | gap_seconds.isNull(), 1).otherwise(0))
    .withColumn("session_id", spark_sum3("is_new_session").over(w.rowsBetween(Window.unboundedPreceding, Window.currentRow)))
)
sessions.orderBy("user_id", "event_time").show(truncate=False)

### Problem 4 — Nth highest value per group

"Find the 2nd highest salary in each department" — the SQL classic, translated directly: `dense_rank()` (not `row_number()`) so that tied top salaries don't push the real 2nd-highest out of position.

In [ ]:
emp_salaries = spark.createDataFrame(
    [
        ("eng", "a", 100), ("eng", "b", 100), ("eng", "c", 90),
        ("sales", "d", 80), ("sales", "e", 70),
    ],
    ["dept", "name", "salary"],
)

w = Window.partitionBy("dept").orderBy(col("salary").desc())
second_highest = (
    emp_salaries
    .withColumn("drnk", dense_rank().over(w))
    .filter(col("drnk") == 2)
    .drop("drnk")
)
second_highest.show()
# eng -> c/90 (the two 100s tie for 1st with dense_rank, so 90 correctly ranks 2nd)

## 4. Final interview checklist

- Can you explain lazy evaluation and why it enables optimization?
- Can you name the physical join strategies and when each is chosen?
- Can you diagnose data skew from the Spark UI and describe 2+ fixes?
- Can you write a top-N-per-group / dedup-latest-record query from memory with `Window` + `row_number`?
- Can you explain `repartition` vs `coalesce`, and when caching helps vs. hurts?
- Can you describe what a watermark does and why streaming aggregations need one?

**Related practice in this repo:** the SQL equivalents of most of these problems (joins, window functions, recursive CTEs, upserts) live in `../sql_postgres_practice/` — worth cross-checking, since interviewers often ask you to translate between SQL and Spark for the same problem.